In [7]:
import torch
from torch.utils import data
import re
from torch import nn
from torch.nn import functional as F
import math
import random
batch_size=32
num_step=35
lr=1
epoch=500

In [8]:
def corpus_vocab_maker():
    with open('../data/timemachine.txt','r') as file:
        text=file.read().lower()
    text=re.sub('[^A-Za-z]+',' ',text).strip()
    text=text.split()
    idx_to_token=[*(['<unk>']+list(set(text)))]
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab={
        "idx_to_token":idx_to_token,
        "token_to_idx":token_to_idx
    }
    corpus=torch.tensor([token_to_idx[token] for token in text])
    return corpus,vocab

In [9]:
def train_iter_maker(corpus,batch_size,num_step):
    t = random.randint(
        0,
        num_step - 1
    )
    corpus = corpus[t:]
    sequence=corpus.unfold(dimension=0,size=num_step+1,step=num_step)
    x = sequence[:, :-1]
    y = sequence[:, 1:]
    dataset=data.TensorDataset(x,y)
    train_iter=data.DataLoader(dataset,batch_size=batch_size,shuffle=True)
    return train_iter

# class SequentialTextLoader:
#     def __init__(
#         self,
#         corpus,
#         batch_size,
#         num_step
#     ):
#         self.corpus = (
#             corpus
#             .flatten()
#             .long()
#         )

#         self.batch_size = batch_size
#         self.num_step = num_step

#     def __iter__(self):
#         # 每次进入一个新epoch时，
#         # 都重新产生随机切片偏移
#         t = random.randint(
#             0,
#             self.num_step - 1
#         )

#         # 减1是因为还需要给X制作错开一位的Y
#         num_tokens = (
#             (
#                 len(self.corpus)
#                 - t
#                 - 1
#             )
#             // self.batch_size
#         ) * self.batch_size

#         if num_tokens <= 0:
#             raise ValueError(
#                 "corpus长度不足以构造一个batch"
#             )

#         x_all = self.corpus[
#             t:
#             t + num_tokens
#         ]

#         y_all = self.corpus[
#             t + 1:
#             t + 1 + num_tokens
#         ]

#         # 每一行都是一条连续的文本序列
#         x_all = x_all.reshape(
#             self.batch_size,
#             -1
#         )

#         y_all = y_all.reshape(
#             self.batch_size,
#             -1
#         )

#         num_batches = (
#             x_all.shape[1]
#             // self.num_step
#         )

#         for start in range(
#             0,
#             num_batches * self.num_step,
#             self.num_step
#         ):
#             x = x_all[
#                 :,
#                 start:start + self.num_step
#             ]

#             y = y_all[
#                 :,
#                 start:start + self.num_step
#             ]

#             yield x, y


# def train_iter_maker(
#     corpus,
#     batch_size,
#     num_step
# ):
#     train_iter = SequentialTextLoader(
#         corpus,
#         batch_size,
#         num_step
#     )

#     return train_iter

In [10]:
corpus,vocab=corpus_vocab_maker()
device=torch.device('cuda:0')
vocab_size=len(vocab['token_to_idx'])



In [11]:
class grunet(nn.Module):
    def __init__(self,input_num,hidden_layer):
        super().__init__()
        self.input_num=input_num
        self.hidden_layer=hidden_layer
        self.gru=nn.GRU(input_size=input_num,hidden_size=hidden_layer)
        self.output=nn.Linear(hidden_layer,vocab_size)
    def forward(self,x,state):
        x=F.one_hot(x.T.long(),num_classes=self.input_num).float()
        gru_result,state=self.gru(x,state)
        gru_result=gru_result.reshape(-1,self.hidden_layer)
        result=self.output(gru_result)
        return result,state
    def get_new_state(self,batch_size):
        return torch.zeros(self.gru.num_layers,batch_size,self.hidden_layer,device=device)

In [12]:
net=grunet(vocab_size,256).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-4)
loss=nn.CrossEntropyLoss()
for i in range(epoch):
    net.train()
    train_iter=train_iter_maker(corpus,batch_size,num_step)
    train_loss=0.0
    train_count=0
    for x,y in train_iter:
        
        trainer.zero_grad()
        x=x.to(device)
        y=y.T.reshape(-1).to(device)
        #print(x.shape[0])
        state=net.get_new_state(x.shape[0])
        logit,state=net(x,state)
        l=loss(logit,y)
        l.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),max_norm=1.0)
        trainer.step()
        train_loss+=l.item()*len(y)
        train_count+=len(y)
    scheduler.step()
    print(f"第{i+1}epoch的loss:{train_loss/train_count},困惑度：{math.exp(train_loss/train_count)}")


        



第1epoch的loss:7.543524387176024,困惑度：1888.4740239055545
第2epoch的loss:6.680359958583473,困惑度：796.6058055112561
第3epoch的loss:6.5635581659123226,困惑度：708.7892026635249
第4epoch的loss:6.457667375222231,困惑度：637.5721041732505
第5epoch的loss:6.424868199149555,困惑度：616.9994911258608
第6epoch的loss:6.403004380598425,困惑度：603.6559283671209
第7epoch的loss:6.372299691550752,困惑度：585.4025273487581
第8epoch的loss:6.356070881215935,困惑度：575.9788154091266
第9epoch的loss:6.337303274328058,困惑度：565.2698763312861
第10epoch的loss:6.331906914328509,困惑度：562.2276923242857
第11epoch的loss:6.303812518604299,困惑度：546.6520634566625
第12epoch的loss:6.275084014631744,困惑度：531.1710060199133
第13epoch的loss:6.247439112637769,困惑度：516.6879493244979
第14epoch的loss:6.220910871130789,困惑度：503.1613384879449
第15epoch的loss:6.218882682995918,困惑度：502.1418668209379
第16epoch的loss:6.178337739240677,困惑度：482.18976468724713
第17epoch的loss:6.164259437821869,困惑度：475.4489130447663
第18epoch的loss:6.135774416801257,困惑度：462.09681102426305
第19epoch的loss:6.114796057614413,困

In [13]:
def predict_text(text,num,vocab,net):
    unk_index = vocab["token_to_idx"]["<unk>"]
    text = re.sub(
    "[^A-Za-z]+",
    " ",
    text
    ).strip().lower().split()
    #text=text.split()
    net.eval()
    output_idx=[vocab['token_to_idx'].get(text[0],unk_index)]
    state=net.get_new_state(1)
    def get_input():
        return torch.tensor(output_idx[-1],device=device).reshape(1,-1)
    with torch.no_grad():
        for token in text[1:]:
            _,state=net(get_input(),state)
            output_idx.append(vocab['token_to_idx'].get(token,unk_index))
        for i in range(num):
            y,state=net(get_input(),state)
            y=y[-1].argmax(dim=-1).item()
            output_idx.append(y)
        return " ".join([
                vocab['idx_to_token'][index]
                for index in output_idx])

In [14]:
predict_text("Time traveller",100,vocab,net)

'time traveller s a time i hesitated and dined again i turned to look at last with my growing so i turned again to myself and dress up and i began to feel it but it seemed to be that night there is no necessity no signs of proprietary rights or the light and my conversational beginnings ended i noted for the first time that s all those who had seen the sun had a small gallery like perhaps a man dance through it appeared to me to you so but it was so that the sun did not only how it'